In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import yaml
import spacy
from corextopic import corextopic as ct
from dvclive import Live
from matplotlib.figure import Figure
from spacy.tokens import DocBin

from job_post_nlp.utils.interactive import try_inter

try_inter()
from job_post_nlp.prepare import corpus_unpack, register_extensions, load_data,register_extensions, load_texts # noqa: E402
from job_post_nlp.utils.find_project_root import find_project_root  # noqa: E402
from job_post_nlp.evaluate import load_model  # noqa: E402
from job_post_nlp.train import load_corpus_split, load_tdm  # noqa: E402
import helpfuncs as hf

/home/b281467@PROD.SITAD.DK/.conda/envs/jobpostnlp/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# store in one dictionary
data = hf.load_everything()
model = data['model']

In [3]:
model.get_top_docs(topic=0, n_docs=10)

[('4021496', np.float64(0.0)),
 ('4021566', np.float64(0.0)),
 ('4516861', np.float64(0.0)),
 ('4021556', np.float64(0.0)),
 ('4021555', np.float64(0.0)),
 ('4516865', np.float64(0.0)),
 ('4021541', np.float64(0.0)),
 ('4021525', np.float64(0.0)),
 ('4021523', np.float64(0.0)),
 ('4021515', np.float64(0.0))]

In [4]:
hf.print_words_in_doc('2837330', data)

god, arbejde, positiv, overblik, søge, person, fremstille, best
western, hotel, jens, baggesen, kok, dygtig, alsidig, kreativ
selvstændig, glad, værdsætte, anden, mening, spændende, velsmagende, mad
selskab, konferencegæst, omhyggelig, køkkenhygiejne, periodevis, travl, periode, varebestilling
sammensætning, menu, sætter, pris, stabilie, kollegaere, god overblik, arbejde selvstændig


In [5]:
hf.print_words_and_text('2837330', data)

Text for document 2837330:
Best Western Hotel Jens Baggesen søger kok *Du er dygtig, alsidig og kreativ kom med godt
 overblik *Kan arbejde selvstændig *En glad og positiv person, der også værdsætter
 andres mening *Arbejde på mindre hotel *Fremstille spændende og velsmagende mad
 til selskaber og konferencegæster *Er omhyggelig med køkkenhygiejne *Periodevis
 travle perioder *Varebestilling og sammensætning af menuer *Sætter pris på gode
 og stabilie kollegaere
Words in document:
god, arbejde, positiv, overblik, søge, person, fremstille, best
western, hotel, jens, baggesen, kok, dygtig, alsidig, kreativ
selvstændig, glad, værdsætte, anden, mening, spændende, velsmagende, mad
selskab, konferencegæst, omhyggelig, køkkenhygiejne, periodevis, travl, periode, varebestilling
sammensætning, menu, sætter, pris, stabilie, kollegaere, god overblik, arbejde selvstændig


In [38]:
hf.print_random(data)


Vacancy ID: 4093269
Text for document 4093269:
Forløbskoordinator Randers Kommune søger koordinator til Aktiv Sygemeldt Har du lysten,
 modet og viljen, så har vi en krævende og udfordrende fuldtidsstilling til dig.
 Randers Kommune søger pr. 1. marts 2016 en forløbskoordinator til Aktiv Sygemeldt.
 Vi søger en forløbskoordinator til Aktiv Sygemeldt, som skal koordinere forløb
 for sygemeldte borgere med en senhjerneskade med henblik på at sikre fastholdelse
 på arbejdsmarkedet. Stillingen som forløbskoordinator i Aktiv Sygemeldt indeholder
 derfor en høj grad af jobkonsulentopgaver. Udover den tætte kontakt til den
 sygemeldte borger, sagsbehandlerne på Jobcentret og arbejdspladserne, er dine samarbejdspartnere
 en bred palet af fagprofessionelle – herunder vores neuroteam
 af fysioterapeuter, ergoterapeuter og forløbskoordinator, neuroundervisere på
 Hjernecentret, sagsbehandlere i Socialafdelingen , Visitationen m.fl. Du skal
 have en solid erfaring og forståelse med arbejdet med m

Text for document 4142621:
Til en af vores kunder, en større produktionsvirksomhed i Århus, søger vi en erfaren
 smed/reparatør til et vikariat på 2-4 måneder, evt. længere. Virksomheden forventer,
 at du har en faglært baggrund som fx smed, maskinarbejder/industrielektriker
 eller noget helt andet. Vigtigst er dog at du har smede- og reparatørerfaring fra et
 tidligere job, da arbejdsopgaverne omfatter både svejsning, hydraulik/pneumatik
 og meget andet: TIG samt MAG svejsning (uden certifikat) Fejlsøgning og reparation
 af pneumatik/hydraulik Opbygning af fixturer til svejse processer Indstillet
 på mange forskellige adhoc-opgaver Vi forventer du generelt er serviceminded.
 Du vil få nogle fine kolleger i værktøjsafdelingen, men som udgangspunkt forventes
 det, at du selvstændig kan klare svejsninger og de fleste reparationer/fejlsøgninger
 m.v. Du skal være indstillet på selv at have ”hænderne på maskinerne" og få "olie
 på fingrene”. Løn efter lokalaftale. Opstart straks, så vent i

In [7]:
ids_sorted = model.word_freq.argsort()
np.array(model.words)[ids_sorted[-10:]]

array(['ansøgning', 'opgave', 'tilbyde', 'samarbejde', 'samt', 'erfaring',
       'stilling', 'god', 'arbejde', 'søge'], dtype='<U100')

In [45]:
hf.print_doc_containing_word('ambitiøs', data)

Documents containing the word 'ambitiøs':

Document 1 (ID: 4943207):
Text for document 4943207:
Arbejdssted: Planafdelingen, By- og Udviklingsforvaltningen, Nytorv 11, 6000 Kolding
 Arbejdstid: 37 timer pr. uge med flekstid Ansættelsestidspunkt: 1. maj 2019
 eller snarest muligt Hvad kan vi tilbyde: Kolding Kommune er inde i en spændende og rivende
 udvikling, hvor Planafdelingen spiller en central rolle i såvel udvikling
 af byerne og landdistrikterne. Vi har en faglig ambition om, at den planlægning vi arbejder
 med bidrager positivt til byernes liv og udvikling samt understøtter liv i
 landdistrikterne – og at vores planlægning har hold i virkeligheden. Planafdelingen
 arbejder med en bred vifte af planlægningsopgaver, herunder planlægning i landzone.
 Derfor tilbyder vi en spændende stilling, hvor du medvirker til at påvirke vores
 arbejde med landdistrikterne, herunder især landzonesagsbehandlingen. Hvis
 du har lyst og er blevet nysgerrig, så send endelig en ansøgning til os. Fun